# Silver resource transformations

Create a current-state Silver snapshot with resource-specific typed columns. Raw JSON is retained to avoid losing nested and polymorphic FHIR fields.

In [ ]:
resource_type = "Patient"
from pyspark.sql import functions as F
from pyspark.sql.window import Window

allowed = {"Patient", "Encounter", "Observation", "Condition"}
if resource_type not in allowed:
    raise ValueError(f"Unsupported resource_type: {resource_type}")
bronze_table = f"bronze_{resource_type.lower()}"
if not spark.catalog.tableExists(bronze_table):
    raise ValueError(f"Missing Bronze table: {bronze_table}")
source = spark.table(bronze_table)

def jpath(path):
    return F.get_json_object(F.col("raw_resource_json"), path)

base = source.select(
    "resource_type","resource_id","source_version_id","source_last_updated",
    "extraction_timestamp","ingestion_timestamp","api_url_or_params","batch_id","raw_resource_json"
)
if resource_type == "Patient":
    silver = base.withColumn("active", jpath("$.active").cast("boolean")) \
        .withColumn("gender", jpath("$.gender")) \
        .withColumn("birth_date", jpath("$.birthDate")) \
        .withColumn("family_name", jpath("$.name[0].family")) \
        .withColumn("given_name", jpath("$.name[0].given[0]")) \
        .withColumn("deceased_boolean", jpath("$.deceasedBoolean").cast("boolean"))
elif resource_type == "Encounter":
    silver = base.withColumn("status", jpath("$.status")) \
        .withColumn("class_code", jpath("$.class.code")) \
        .withColumn("period_start", jpath("$.period.start")) \
        .withColumn("period_end", jpath("$.period.end")) \
        .withColumn("patient_reference", jpath("$.subject.reference"))
elif resource_type == "Observation":
    silver = base.withColumn("status", jpath("$.status")) \
        .withColumn("code", jpath("$.code.coding[0].code")) \
        .withColumn("code_display", jpath("$.code.coding[0].display")) \
        .withColumn("patient_reference", jpath("$.subject.reference")) \
        .withColumn("encounter_reference", jpath("$.encounter.reference")) \
        .withColumn("effective_datetime", jpath("$.effectiveDateTime")) \
        .withColumn("value_quantity", jpath("$.valueQuantity.value").cast("double")) \
        .withColumn("value_unit", jpath("$.valueQuantity.unit")) \
        .withColumn("value_string", jpath("$.valueString"))
else:
    silver = base.withColumn("clinical_status", jpath("$.clinicalStatus.coding[0].code")) \
        .withColumn("verification_status", jpath("$.verificationStatus.coding[0].code")) \
        .withColumn("code", jpath("$.code.coding[0].code")) \
        .withColumn("code_display", jpath("$.code.coding[0].display")) \
        .withColumn("patient_reference", jpath("$.subject.reference")) \
        .withColumn("encounter_reference", jpath("$.encounter.reference")) \
        .withColumn("onset_datetime", jpath("$.onsetDateTime"))

# Keep the latest source version for each logical resource in the current Silver snapshot.
w = Window.partitionBy("resource_id").orderBy(
    F.col("source_last_updated").desc_nulls_last(),
    F.col("source_version_id").desc_nulls_last(),
    F.col("ingestion_timestamp").desc()
)
silver = (silver.withColumn("_rn", F.row_number().over(w))
          .filter(F.col("_rn") == 1).drop("_rn"))
table_name = f"silver_{resource_type.lower()}"
(silver.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(table_name))
print(f"Wrote {silver.count()} current resources to {table_name}; raw JSON retained.")
